# HateXplain dataset — setup

Load the raw JSON, inspect one post, then summarize labels and annotator agreement.


## 1. Load Dataset

Read `dataset.json` into memory and confirm it loaded.


### Imports

We only need the standard library here.


In [1]:
import json



### Load JSON

Path is relative to this notebook (`notebooks/`).


In [2]:
with open("../data/raw/hatexplain/dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Loaded successfully")

Loaded successfully


### Quick confirmation

Type and number of top-level entries.


In [ ]:
print(type(data))
print(len(data))

<class 'dict'>
20148


## 2. Inspect Dataset Structure

The file is one big dictionary: each **key** is a post ID, each **value** is a post object.


### Example post IDs

Keys are strings (often `tweet_id_platform`).


In [9]:
keys = list(data.keys())
print(keys[:5])

['1179055004553900032_twitter', '1179063826874032128_twitter', '1178793830532956161_twitter', '1179088797964763136_twitter', '1179085312976445440_twitter']


### One post as a Python dict

We grab the first key and print the whole value.


In [16]:
sample = data[keys[0]]
print(sample)

{'post_id': '1179055004553900032_twitter', 'annotators': [{'label': 'normal', 'annotator_id': 1, 'target': ['None']}, {'label': 'normal', 'annotator_id': 2, 'target': ['None']}, {'label': 'normal', 'annotator_id': 3, 'target': ['None']}], 'rationales': [], 'post_tokens': ['i', 'dont', 'think', 'im', 'getting', 'my', 'baby', 'them', 'white', '9', 'he', 'has', 'two', 'white', 'j', 'and', 'nikes', 'not', 'even', 'touched']}


### Keys inside one post

Every value should expose the same top-level fields.


In [ ]:
# sample_key = list(data.keys())[1]
# sample = data[sample_key]

print("Keys inside one sample:")
print(sample.keys())

Keys inside one sample:
dict_keys(['post_id', 'annotators', 'rationales', 'post_tokens'])


### Reconstruct text from `post_tokens`

Tokens are stored as a list; join with spaces to read the post.


In [18]:
tokens = sample["post_tokens"]
text = " ".join(tokens)

print(text)

i dont think im getting my baby them white 9 he has two white j and nikes not even touched


### Annotator entries

Each post has three annotators with `label`, `annotator_id`, and `target`.


In [22]:
print(sample["annotators"])

[{'label': 'normal', 'annotator_id': 1, 'target': ['None']}, {'label': 'normal', 'annotator_id': 2, 'target': ['None']}, {'label': 'normal', 'annotator_id': 3, 'target': ['None']}]


### More raw examples (first 5 posts)

Full dictionaries including `rationales` when present — useful for seeing the JSON shape.


In [20]:
# Show 5 samples instead of 1
for i, (key, value) in enumerate(data.items()):
    print(f"Sample ID: {key}")
    print(value)
    print("-" * 50)
    
    if i == 4:  # stop after 5 samples
        break

Sample ID: 1179055004553900032_twitter
{'post_id': '1179055004553900032_twitter', 'annotators': [{'label': 'normal', 'annotator_id': 1, 'target': ['None']}, {'label': 'normal', 'annotator_id': 2, 'target': ['None']}, {'label': 'normal', 'annotator_id': 3, 'target': ['None']}], 'rationales': [], 'post_tokens': ['i', 'dont', 'think', 'im', 'getting', 'my', 'baby', 'them', 'white', '9', 'he', 'has', 'two', 'white', 'j', 'and', 'nikes', 'not', 'even', 'touched']}
--------------------------------------------------
Sample ID: 1179063826874032128_twitter
{'post_id': '1179063826874032128_twitter', 'annotators': [{'label': 'normal', 'annotator_id': 1, 'target': ['None']}, {'label': 'normal', 'annotator_id': 2, 'target': ['None']}, {'label': 'normal', 'annotator_id': 3, 'target': ['None']}], 'rationales': [], 'post_tokens': ['we', 'cannot', 'continue', 'calling', 'ourselves', 'feminists', 'if', 'the', 'rights', 'of', 'all', 'womxn', 'arent', 'addressed', 'yes', 'to', 'a', 'sexual', 'offences', '

## 3. Initial Label Analysis

**Final label** = majority vote across the three annotators (ties broken by `Counter.most_common`).


### Majority vote on the sample post

Same post as above: three votes and the winning label.


In [23]:
from collections import Counter

labels = [ann["label"] for ann in sample["annotators"]]
final_label = Counter(labels).most_common(1)[0][0]

print("Labels:", labels)
print("Final label:", final_label)

Labels: ['normal', 'normal', 'normal']
Final label: normal


### Unique final labels

Run majority vote on every post and collect the distinct outcomes.


In [24]:
all_labels = []

for item in data.values():
    labels = [ann["label"] for ann in item["annotators"]]
    final_label = Counter(labels).most_common(1)[0][0]
    all_labels.append(final_label)

print(set(all_labels))

{'offensive', 'normal', 'hatespeech'}


### Counts per final label

How many posts land in each class after majority vote.


In [25]:
from collections import Counter

final_labels = []

for item in data.values():
    labels = [ann["label"] for ann in item["annotators"]]
    final_label = Counter(labels).most_common(1)[0][0]
    final_labels.append(final_label)

print(Counter(final_labels))

Counter({'normal': 8153, 'hatespeech': 6234, 'offensive': 5761})


### Annotator agreement

- **Full agreement**: all three chose the same label.  
- **2-vs-1**: two labels match, one differs.  
- **All different**: three distinct labels.


In [26]:
disagreement_count = 0
full_agreement_count = 0
undecided_count = 0

for item in data.values():
    labels = [ann["label"] for ann in item["annotators"]]
    unique_labels = set(labels)

    if len(unique_labels) == 1:
        full_agreement_count += 1
    elif len(unique_labels) == 2:
        disagreement_count += 1
    elif len(unique_labels) == 3:
        undecided_count += 1

print("Full agreement:", full_agreement_count)
print("2-vs-1 disagreement:", disagreement_count)
print("All different:", undecided_count)

Full agreement: 9845
2-vs-1 disagreement: 9384
All different: 919


### Empty or whitespace-only text

Sanity check on reconstructed text.


In [27]:
empty_text_count = 0

for item in data.values():
    text = " ".join(item["post_tokens"]).strip()
    if text == "":
        empty_text_count += 1

print("Empty texts:", empty_text_count)

Empty texts: 0


### First five posts: text, votes, and final label

Skim reconstructed text next to per-annotator labels and the majority result.


In [28]:
for i, item in enumerate(data.values()):
    text = " ".join(item["post_tokens"])
    labels = [ann["label"] for ann in item["annotators"]]
    final_label = Counter(labels).most_common(1)[0][0]

    print("Text:", text)
    print("Annotator labels:", labels)
    print("Final label:", final_label)
    print("-" * 60)

    if i == 4:
        break

Text: i dont think im getting my baby them white 9 he has two white j and nikes not even touched
Annotator labels: ['normal', 'normal', 'normal']
Final label: normal
------------------------------------------------------------
Text: we cannot continue calling ourselves feminists if the rights of all womxn arent addressed yes to a sexual offences public list but will a trans lesbian bisexual and queer womxn be able to enter their information on the reporting sheet gender forum
Annotator labels: ['normal', 'normal', 'normal']
Final label: normal
------------------------------------------------------------
Text: nawt yall niggers ignoring me
Annotator labels: ['normal', 'normal', 'hatespeech']
Final label: normal
------------------------------------------------------------
Text: <user> i am bit confused coz chinese ppl can not access twitter thn how this ching chong using it i think he pakistani 🤔 🤔 🤔
Annotator labels: ['hatespeech', 'offensive', 'hatespeech']
Final label: hatespeech
----

## 4. Preprocessing and Data Structuring

Turn the raw JSON into a training-ready table: text, numeric labels, and flags for annotator agreement.


### Step 4.1 — Build Structured DataFrame

Convert raw JSON data into a structured DataFrame with labels, agreement information, and reconstructed text.


In [29]:
import pandas as pd
from collections import Counter

rows = []

for post_id, item in data.items():
    annotator_labels = [ann["label"] for ann in item["annotators"]]
    label_counts = Counter(annotator_labels)

    unique_count = len(set(annotator_labels))
    is_full_agreement = 1 if unique_count == 1 else 0
    is_disagreement = 1 if unique_count == 2 else 0
    is_undecided = 1 if unique_count == 3 else 0

    final_label = None
    if not is_undecided:
        final_label = label_counts.most_common(1)[0][0]

    text = " ".join(item["post_tokens"]).strip()

    if "_twitter" in post_id:
        source = "twitter"
    elif "_gab" in post_id:
        source = "gab"
    else:
        source = "unknown"

    targets = []
    for ann in item["annotators"]:
        if "target" in ann and ann["target"] is not None:
            targets.extend(ann["target"])
    targets = list(set(targets))

    rows.append({
        "post_id": post_id,
        "text": text,
        "annotator_labels": annotator_labels,
        "label_counts": dict(label_counts),
        "final_label": final_label,
        "is_full_agreement": is_full_agreement,
        "is_disagreement": is_disagreement,
        "is_undecided": is_undecided,
        "targets": targets,
        "rationales": item.get("rationales", []),
        "source": source
    })

df = pd.DataFrame(rows)

print(df.head())
print(df.shape)


                       post_id  \
0  1179055004553900032_twitter   
1  1179063826874032128_twitter   
2  1178793830532956161_twitter   
3  1179088797964763136_twitter   
4  1179085312976445440_twitter   

                                                text  \
0  i dont think im getting my baby them white 9 h...   
1  we cannot continue calling ourselves feminists...   
2                      nawt yall niggers ignoring me   
3  <user> i am bit confused coz chinese ppl can n...   
4  this bitch in whataburger eating a burger with...   

                      annotator_labels                       label_counts  \
0             [normal, normal, normal]                      {'normal': 3}   
1             [normal, normal, normal]                      {'normal': 3}   
2         [normal, normal, hatespeech]     {'normal': 2, 'hatespeech': 1}   
3  [hatespeech, offensive, hatespeech]  {'hatespeech': 2, 'offensive': 1}   
4  [hatespeech, hatespeech, offensive]  {'hatespeech': 2, 'offensive': 1}

### Step 4.2 — Remove Undecided Samples

Remove samples where all annotators disagree (no majority label).


In [30]:
df_clean = df[df["is_undecided"] == 0].copy()

print("Original shape:", df.shape)
print("Clean shape:", df_clean.shape)
print("Removed undecided:", df["is_undecided"].sum())


Original shape: (20148, 11)
Clean shape: (19229, 11)
Removed undecided: 919


### Step 4.3 — Encode Labels

Convert textual labels into numeric IDs for model training.


In [31]:
label_map = {
    "hatespeech": 0,
    "offensive": 1,
    "normal": 2
}

df_clean["label_id"] = df_clean["final_label"].map(label_map)

print(df_clean[["final_label", "label_id"]].head())
print(df_clean["label_id"].value_counts())


  final_label  label_id
0      normal         2
1      normal         2
2      normal         2
3  hatespeech         0
4  hatespeech         0
label_id
2    7814
0    5935
1    5480
Name: count, dtype: int64


### Step 4.4 — Data Validation

Ensure there are no missing or broken values after preprocessing.


In [32]:
print("Null texts:", df_clean["text"].isna().sum())
print("Empty texts:", (df_clean["text"].str.strip() == "").sum())
print("Null labels:", df_clean["final_label"].isna().sum())
print("Null label_ids:", df_clean["label_id"].isna().sum())


Null texts: 0
Empty texts: 0
Null labels: 0
Null label_ids: 0


### Step 4.5 — Final Label Distribution

Check class distribution after cleaning.


In [33]:
print(df_clean["final_label"].value_counts())
print(df_clean["final_label"].value_counts(normalize=True) * 100)


final_label
normal        7814
hatespeech    5935
offensive     5480
Name: count, dtype: int64
final_label
normal        40.636539
hatespeech    30.864840
offensive     28.498622
Name: proportion, dtype: float64


### Step 4.6 — Agreement Analysis

Keep disagreement information for later explainability analysis.


In [34]:
print("Full agreement:", df_clean["is_full_agreement"].sum())
print("Disagreement:", df_clean["is_disagreement"].sum())


Full agreement: 9845
Disagreement: 9384


### Step 4.7 — Save Processed Dataset

Save the cleaned dataset for later training.


In [35]:
import os
os.makedirs("../data/processed", exist_ok=True)

df_clean.to_csv("../data/processed/hatexplain_preprocessed.csv", index=False)

print("Saved successfully.")


Saved successfully.
